In [ ]:
import pymysql
import pandas as pd

# 建立连接（你可以替换自己的连接信息）
conn = pymysql.connect(
    host='###',
    port=3306,
    user='###',
    password='###',
    database='ch00',
    charset='utf8'
)
query_full = """
SELECT dt, ticker, share, close, last_hold
FROM ppLog
WHERE boid = 43
  AND dt BETWEEN '2024-03-01' AND '2025-12-31'
ORDER BY dt, ticker
"""
df_full = pd.read_sql(query_full, conn)
print(df_full)

In [ ]:
# 按 ticker 排序后，对每只股票计算价格变动
df_full = df_full.sort_values(['ticker', 'dt']).copy()

df_full['close_shift'] = df_full.groupby('ticker')['close'].shift(1)
df_full['price_change'] = df_full['close'] - df_full['close_shift']

df_full['return'] = df_full['price_change'] / df_full['close_shift']
df_long = df_full[df_full['last_hold'] > 0].copy()
df_long['value'] = df_long['last_hold'] * df_long['close']

# 总市值
total_value = df_long.groupby('dt')['value'].sum().rename('total_value')
df_long = df_long.merge(total_value, on='dt')
print(df_long)

In [ ]:
df_long['weight'] = df_long['value'] / df_long['total_value']
df_long['weighted_return'] = df_long['weight'] * df_long['return']
portfolio_returns = df_long.groupby('dt')['weighted_return'].sum().sort_index()
print(portfolio_returns)
print(total_value)

In [ ]:
import matplotlib.pyplot as plt

cumulative_return = (1 + portfolio_returns).cumprod()-1

plt.figure(figsize=(14, 6))
plt.plot(cumulative_return.index, cumulative_return.values, label='return', marker='o')

plt.title("return curve")
plt.xlabel("date")
plt.ylabel("return")
plt.xticks(rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 示例 SQL 查询（你可以替换为你的实际数据库操作）
query = """
SELECT dt, last, diff
FROM rt
WHERE ticker = '000300.SH' and dt>'20240301'
ORDER BY dt ASC
"""

# 读取数据
df_index = pd.read_sql(query, conn)  # conn 是你的数据库连接
print(df_index)


In [ ]:
df_index = df_index.sort_values('dt').copy()  # 按日期排序
df_index['last_shift'] = df_index['last'].shift(1)  # 前一日的 last

# 计算每日收益率（注意第一天结果为 NaN）
df_index['index_return'] = df_index['diff'] / df_index['last_shift']


df_index['dt'] = pd.to_datetime(df_index['dt'])  # 转换为 datetime 类型
df_index['dt'] = df_index['dt'].dt.normalize()  # 保留日期，丢掉时间部分，但仍是 datetime 类型

df_index = df_index.set_index('dt')  # 正确设置 index

# 计算累计收益率（从第二天开始）
df_index['index_cum_return'] = (1 + df_index['index_return']).cumprod()-1

print(df_index)
print(df_index.index)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(cumulative_return.index, cumulative_return.values, label='portfolio return', marker='o')
plt.plot(df_index.index, df_index['index_cum_return'], marker='s', label='index return')

plt.title('中证300指数累计收益率曲线')
plt.xlabel('日期')
plt.ylabel('累计收益率')
plt.xticks(rotation=45)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 如果还不是 DataFrame，先将 portfolio_returns 转换为 DataFrame
df_portfolio = portfolio_returns.to_frame(name='portfolio_return')


# 合并两个 DataFrame（按索引对齐）
df_combined = df_portfolio.join(df_index[['index_return']])

# 计算超额收益率
df_combined['excess_return'] = df_combined['portfolio_return'] - df_combined['index_return']

# 计算超额收益率的累积收益曲线
df_combined['excess_cum_return'] = (1 + df_combined['excess_return']).cumprod()-1

# 打印结果确认
print(df_combined)

In [ ]:
plt.figure(figsize=(12, 6))

plt.plot(df_combined.index, df_combined['excess_cum_return'], label='excess return', marker='o')
plt.plot(cumulative_return.index, cumulative_return.values, label='portfolio return', marker='o')
plt.plot(df_index.index, df_index['index_cum_return'], marker='s', label='index return')

plt.xlabel('日期')
plt.ylabel('累积超额收益率')
plt.title('excess return')
plt.grid(True)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
conn = pymysql.connect(host='###', user='###', password='###', db='ch00')

query = """
SELECT DISTINCT ticker
FROM pplog
WHERE boid = 43
  AND dt BETWEEN '2023-08-25' AND '2025-12-31';
"""

df_candidate_tickers = pd.read_sql(query, conn)
tickers = df_candidate_tickers['ticker'].unique().tolist()
print(len(tickers))



In [ ]:

# 构建 ticker 列表字符串
tickers_str = ','.join(f"'{t}'" for t in tickers)
print(tickers_str)

query = f"""
SELECT locdt, ticker, last, diff
FROM rt
WHERE ticker IN ({tickers_str})
  AND locdt BETWEEN '2023-08-25' AND '2025-06-09';
"""

df_price = pd.read_sql(query, conn)

# 重命名列
df_price = df_price.rename(columns={
    'locdt': 'dt',
    'last': 'close',
    'diff': 'diff'  # 保持 diff 不变
})

# 转换为日期格式（如需要）
df_price['dt'] = pd.to_datetime(df_price['dt'])

# 可选：按日期排序，检查数据
df_price = df_price.sort_values(['dt', 'ticker'])
print(df_price.head())
print(df_price.shape)

In [ ]:
query = """
SELECT dt, ticker
FROM pplog
WHERE boid = 43
  AND dt BETWEEN '2023-08-25' AND '2025-06-09';
"""

df_holdings = pd.read_sql(query, conn)
df_holdings['dt'] = pd.to_datetime(df_holdings['dt'])

# 检查前几天的持仓情况
print(df_holdings.groupby('dt').ticker.apply(list).head())


In [ ]:
import cvxpy as cp
import numpy as np

def optimization(returns_df):
    """
    输入:
        returns_df: 每日收益率数据，行为时间，列为股票代码
    输出:
        Series：最优权重，索引为股票代码
    """
    n = returns_df.shape[1]
    tickers = returns_df.columns.tolist()
    
    mu = returns_df.iloc[-20:].mean().values       # 期望收益向量 μ
    Sigma = returns_df.cov().values     # 协方差矩阵 Σ
    

    # 定义变量
    w = cp.Variable(n)

    # 构造目标函数: μᵀw - (1/2) * wᵀΣw
    risk = cp.sqrt(cp.quad_form(w, Sigma))
    expected_return = mu @ w
    risk_penalty = cp.quad_form(w, Sigma)
    risk_limit=0.004

    objective = cp.Maximize(expected_return)

    # 约束条件
    constraints = [
        cp.sum(w) == 1,  # 权重和为1
        cp.quad_form(w, Sigma) <= risk_limit,
        w >= 0,           # 不允许做空
        w<=0.05,
    ]

    # 求解
    problem = cp.Problem(objective, constraints)
    problem.solve()

    # 返回结果
    if w.value is not None:
        return pd.Series(w.value, index=tickers)
    else:
        print("⚠️ 优化失败")
        return pd.Series([np.nan] * n, index=tickers)


In [ ]:
def optimization_from_mu_sigma(mu, Sigma):
    n = len(mu)
    w = cp.Variable(n)
    
    expected_return = mu @ w
    risk_penalty = cp.quad_form(w, Sigma)
    objective = cp.Maximize(expected_return)

    constraints = [
        cp.sum(w) == 1,
        cp.quad_form(w, Sigma) <= 0.006,
        w >= 0,
        w <= 0.06
    ]

    prob = cp.Problem(objective, constraints)
    prob.solve(solver=cp.MOSEK)

    if w.value is not None:
        return pd.Series(w.value, index=tickers)
    else:
        print("⚠️ 优化失败")
        return pd.Series([np.nan] * n, index=tickers)

In [ ]:
def clip_daily_percentile(df, lower_pct, upper_pct):
    """
    对 DataFrame 按行（每天）进行 clip：每一行保留中间百分比的值，将极端值限制在分位数范围内。
    """
    q_low = df.quantile(lower_pct, axis=1)
    q_high = df.quantile(upper_pct, axis=1)

    # 对每一行执行 clip，用 apply 和 zip 逐行处理
    df_clipped = df.copy()

    for i in df.index:
        df_clipped.loc[i] = df.loc[i].clip(lower=q_low.loc[i], upper=q_high.loc[i])

    return df_clipped

In [ ]:
df_price_wide = df_price.pivot(index='dt', columns='ticker', values='close')

# 确保 df_price_wide 的 index 是按时间升序的 DatetimeIndex
all_days = df_price_wide.index.sort_values()
all_days = all_days[all_days != pd.Timestamp('2025-01-01')]

# 从第40天开始，每隔10天取一个作为调仓日
lookback_days_var = 120
holding_days = 1

# e.g. 从第40个交易日开始，每隔10天取一个日期
target_days = all_days[lookback_days_var::holding_days]
opt_portfolio_returns=[]
rebalance_days = []
i=0
for target_day in target_days:
    rebalance_days.append(target_day)
    tickers_today=df_holdings[df_holdings['dt']==target_day]['ticker'].unique().tolist()
    window_days = all_days[all_days<=target_day][-lookback_days_var:]
    df_window = df_price_wide.loc[window_days, tickers_today]
    returns_df = df_window.pct_change().dropna()


    #排除收益最高股票
    '''last_day = window_days[-1]
    prev_day = window_days[-20]
    prev_prices = df_price_wide.loc[prev_day, tickers_today]
    curr_prices = df_price_wide.loc[last_day, tickers_today]
    daily_return = (curr_prices - prev_prices) / prev_prices
    excluded_tickers=daily_return[daily_return>=0.20].index.tolist()
    # 找到涨幅最大的股票
    top_rising_ticker = daily_return.idxmax()
    returns_df=returns_df.drop(columns=excluded_tickers,errors='ignore')
    tickers_today = returns_df.columns.tolist()'''



    sigma = returns_df.cov().values

    '''returns_clipped = returns_df.clip(upper=0.10,lower=-0.10)
    mu = returns_clipped.iloc[-20:].mean().values'''

    returns_recent = returns_df.iloc[-20:]  # 只取近20天作为 mu 基础
    '''returns_clipped = clip_daily_percentile(returns_recent, 0.0, 1)
    returns_clipped = returns_df.clip(upper=0.15,lower=-0.15)
    mu = returns_clipped.mean().values'''

        # 获取前两天的价格
    prev2_day = window_days[-3]
    prev1_day = window_days[-2]

    price_t_2 = df_price_wide.loc[prev2_day, tickers_today]
    price_t_1 = df_price_wide.loc[prev1_day, tickers_today]

    # 计算两天价格比的收益率（不用 .pct_change() 避免多余步骤）
    mu = (price_t_1 / price_t_2 - 1).values  # shape = (n_stocks,)




    tickers = returns_df.columns.tolist()
    weights = optimization_from_mu_sigma(mu,sigma)
    i+=1

    # 计算未来 holding_days 的收益
    next_days = all_days[all_days >= target_day][:holding_days+1]
    if len(next_days) < holding_days:
        continue

    future_prices = df_price_wide.loc[next_days, tickers_today]

    future_returns = future_prices.pct_change().dropna()

    ##future_returns = future_returns.drop(columns=excluded_tickers, errors='ignore')
    if i ==1:
        print(future_returns.shape)
        print(weights.shape)
        print(weights.sort_values(ascending=False).head(10))
    
    portfolio_ret = future_returns @ weights
    

    opt_portfolio_returns.append(portfolio_ret)
print(len(opt_portfolio_returns))

opt_portfolio_returns=pd.concat(opt_portfolio_returns)
print(opt_portfolio_returns)

In [ ]:
# 计算累计收益（假设初始为1）
cumulative_returns = (1 + opt_portfolio_returns).cumprod()-1

# 开始绘图
plt.figure(figsize=(20, 6))
plt.plot(cumulative_returns.index, cumulative_returns.values, label='Optimized Portfolio',marker='o')
plt.plot(df_index.index, df_index['index_cum_return'], marker='s', label='index return')
plt.plot(cumulative_return.index, cumulative_return.values, label='return', marker='o')
# 标出调仓日
for day in rebalance_days:
    if day in cumulative_returns.index:  # 避免画不存在的点
        plt.axvline(x=day, color='red', linestyle='--', alpha=0.5)

plt.title("优化组合累计收益曲线", fontsize=14)
plt.xlabel("日期", fontsize=12)
plt.ylabel("累计收益", fontsize=12)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# 总收益（区间最后一天的累计收益）
total_return_optimized = cumulative_returns.iloc[-1]
total_return_orginal= cumulative_return.iloc[-1]

# 收益波动率（基于每日收益）
volatility_optimized = cumulative_returns.std()
volatility_orginal = cumulative_return.std()

# 夏普比率（非年化）
sharpe_ratio_opt = (total_return_optimized-0.05) / volatility_optimized
sharpe_ratio_org = (total_return_orginal-0.05) / volatility_orginal
print("Sharpe Ratio (not annualized):", sharpe_ratio_opt)
print(sharpe_ratio_org)
print(total_return_optimized)

